In [1]:
# ============================================
# CUSTOMER SEGMENTATION - DATA CLEANING
# ============================================

import pandas as pd
import numpy as np
import os

print("Loading dataset...")

# Load Excel dataset
df = pd.read_excel("../data/raw/Online Retail.xlsx")

print("Dataset loaded successfully!")
print("=" * 60)

# Basic information
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

Loading dataset...
Dataset loaded successfully!
Dataset Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [2]:
# ============================================
# DATA CLEANING
# ============================================

# Remove rows without CustomerID
df = df.dropna(subset=["CustomerID"])

# Remove cancelled transactions
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# Remove invalid quantities
df = df[df["Quantity"] > 0]

# Remove invalid prices
df = df[df["UnitPrice"] > 0]

# Convert CustomerID to integer
df["CustomerID"] = df["CustomerID"].astype(int)

# Convert InvoiceDate to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Create transaction amount
df["Amount"] = df["Quantity"] * df["UnitPrice"]

print("Cleaning completed!")
print("=" * 60)
print("Cleaned Shape:", df.shape)
print("Unique Customers:", df["CustomerID"].nunique())
print("Total Revenue: £{:,.2f}".format(df["Amount"].sum()))

Cleaning completed!
Cleaned Shape: (397884, 9)
Unique Customers: 4338
Total Revenue: £8,911,407.90


In [3]:
# ============================================
# PREPARE DATA FOR RFM ANALYSIS
# ============================================

customer_transactions = df.rename(
    columns={
        "InvoiceDate": "TransactionDate"
    }
)

# Select useful columns
customer_transactions = customer_transactions[
    [
        "InvoiceNo",
        "CustomerID",
        "TransactionDate",
        "Quantity",
        "UnitPrice",
        "Amount",
        "Country"
    ]
]

print("Final dataset:")
display(customer_transactions.head())

print("\nFinal columns:")
print(customer_transactions.columns.tolist())

Final dataset:


,InvoiceNo,CustomerID,TransactionDate,Quantity,UnitPrice,Amount,Country
0,536365,17850,2010-12-01 08:26:00,6,2.55,15.30,United Kingdom
1,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,United Kingdom
2,536365,17850,2010-12-01 08:26:00,8,2.75,22.00,United Kingdom
3,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,United Kingdom
4,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,United Kingdom



Final columns:
['InvoiceNo', 'CustomerID', 'TransactionDate', 'Quantity', 'UnitPrice', 'Amount', 'Country']


In [4]:
# ============================================
# SAVE CLEAN DATASET
# ============================================

output_path = "../data/raw/customer_transactions.csv"

customer_transactions.to_csv(
    output_path,
    index=False
)

print("CSV CREATED SUCCESSFULLY!")
print(output_path)
print("Rows:", len(customer_transactions))
print("Customers:", customer_transactions["CustomerID"].nunique())

CSV CREATED SUCCESSFULLY!
../data/raw/customer_transactions.csv
Rows: 397884
Customers: 4338
